In [1]:
!python --version

Python 3.13.15


In [2]:
# install python packages
!pip install ollama requests
!pip install langchain-ollama
!pip install -q langchain-ollama
!pip install -q python-docx
!pip install -q requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.1 MB/s eta 0:00:00


In [3]:
# imports
import json
import requests
from google.colab import userdata
from google.colab import files
from langchain_ollama import ChatOllama
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    ToolMessage
)
import os
from docx import Document
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Load Ollama API key from Colab Secrets
API_KEY = userdata.get("OLLAMA")

if not API_KEY:
    raise ValueError("OLLAMA API key was not found in Colab Secrets.")

# Put it in the environment for the app
os.environ["OLLAMA"] = API_KEY


llm = ChatOllama(
    model="gpt-oss:20b",
    base_url="https://ollama.com",
    temperature=0,
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {API_KEY}"
        },
        "timeout": 30
    }
)

print("Ollama client ready.")

Ollama client ready.


In [5]:
# test the llm
response = llm.invoke("Explain SMB signing in one sentence")
print(response.content)

SMB signing is a security feature that appends a cryptographic signature to every SMB packet, ensuring its integrity and authenticity so that tampering or spoofing can be detected.


In [6]:
# upload the windows environment doc file
uploaded = files.upload()

Saving Windows_Enterprise_Environment.docx to Windows_Enterprise_Environment.docx


In [7]:
# read the word document

# Get the uploaded filename
filename = list(uploaded.keys())[0]

# Read the document
doc = Document(filename)

# Extract paragraphs and tables
passages = []

for paragraph in doc.paragraphs:
    if paragraph.text.strip():
        passages.append(paragraph.text.strip())

for table in doc.tables:
    for row in table.rows:
        text = " | ".join(cell.text.strip() for cell in row.cells)
        if text.strip():
            passages.append(text)

print(f"Loaded {len(passages)} passages from {filename}")

Loaded 125 passages from Windows_Enterprise_Environment.docx


In [8]:
# get relevant smb guidance / information from Microsoft

guidance_urls = [
    "https://learn.microsoft.com/en-us/windows-server/storage/file-server/smb-signing-overview",
    "https://learn.microsoft.com/en-us/windows-server/storage/file-server/smb-security-hardening"
]

guidance_passages = []

for url in guidance_urls:
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        article = soup.find("main")

        if article is None:
            print(f"No article content found: {url}")
            continue

        for element in article.find_all(["p", "li"]):
            text = element.get_text(" ", strip=True)

            if len(text) > 60:
                guidance_passages.append({
                    "text": text,
                    "source": url,
                    "retrieved_at": datetime.now(timezone.utc).isoformat(),
                    "source_mode": "live"
                })

        print(f"Loaded: {url}")

    except requests.RequestException as e:
        print(f"Failed to retrieve {url}: {e}")

print(f"Total guidance passages: {len(guidance_passages)}")

Loaded: https://learn.microsoft.com/en-us/windows-server/storage/file-server/smb-signing-overview
Loaded: https://learn.microsoft.com/en-us/windows-server/storage/file-server/smb-security-hardening
Total guidance passages: 70


In [9]:
# this function was coded with assistance from ChatGPT 6.0 on 10/12/2026 at 5:00PM.
# We defined the function's purpose, inputs, and expected output and added the additional functionality of querying from Microsoft.

def rag_search(
    query: str,
    collection: str = "environment",
    top_k: int = 5
) -> dict:
    """
    Search Northbridge environment evidence or external SMB guidance.

    Args:
        query: Search terms or question.
        collection: 'environment' or 'guidance'.
        top_k: Maximum number of passages returned.

    Returns:
        Dictionary containing the search status, query, collection,
        and matching evidence passages with source attribution
        and relevance scores.
    """

    if collection == "environment":
        documents = [
            {
                "text": text,
                "source": filename,
                "source_mode": "simulated"
            }
            for text in passages
        ]

    elif collection == "guidance":
        documents = guidance_passages

    else:
        return {"status": "error", "message": "Invalid collection"}

    if not query.strip() or not isinstance(top_k, int) or not 1 <= top_k <= 10:
        return {"status": "error", "message": "Invalid query or top_k"}

    if not documents:
        return {"status": "insufficient_evidence", "results": []}

    texts = [doc["text"] for doc in documents]

    vectorizer = TfidfVectorizer(stop_words="english")
    vectors = vectorizer.fit_transform(texts + [query])

    scores = cosine_similarity(
        vectors[-1], vectors[:-1]
    ).flatten()

    ranked = scores.argsort()[::-1][:top_k]

    results = [
        {
            **documents[i],
            "relevance_score": round(float(scores[i]), 4)
        }
        for i in ranked if scores[i] > 0
    ]

    return {
        "status": "success" if results else "no_match",
        "query": query,
        "collection": collection,
        "results": results
    }

In [10]:
# test the external rag search tool
result = rag_search(
    query="How does SMB signing prevent relay and man in the middle attacks?",
    collection="guidance",
    top_k=3
)

for item in result["results"]:
    print("Evidence:", item["text"])
    print("Source:", item["source"])
    print("-" * 50)

Evidence: SMB signing is enabled in all versions of Windows. SMB signing requirements can involve both outbound signing, which covers traffic from the SMB client, and inbound signing, which covers traffic to the server. Windows and Windows Server can require outbound signing only, inbound signing only, both, or neither. If your environment uses non-Microsoft file servers, your system settings can prevent the default settings and connections from taking effect. In this case, you might need to disable the requirement for SMB signing. See Control SMB signing behavior for guidance on how to disable SMB signing.
Source: https://learn.microsoft.com/en-us/windows-server/storage/file-server/smb-signing-overview
--------------------------------------------------
Evidence: All cryptographic keys used in SMB 2.x and 3.x signing are derived from the session key. The security of SMB 2/3 signing and encryption relies on the session key. If someone changes a message during transmission, the hash does

In [11]:
# test the internal rag-search tool
result = rag_search("FILE04 SMB signing")

for item in result["results"]:
    print(item["text"])
    print("---")

W05 File servers | 4 | Department shares and shared documents. SMB 3; SMB1 disabled. Finance and HR shares use AD groups. SMB signing required on three hosts; not required on FILE04.
---
S03 control and exception register | SMB signing is not required on FILE04 within W05; SMB1 disabled across W05. Twelve E02 sensors stale; ten E02 devices have local admin exception. Overlap between these subsets is unknown.
---
Could a reported SMB-related attack apply to FILE04, given signing is not required and SMB1 is disabled? What would access to its data imply? Could a VPN advisory affect N02 when firmware is unknown? What additional evidence is decisive? How do W15 service-account rights and finance data change impact? These are investigation prompts, not established vulnerabilities or answer keys.
---
S02 network design and reviewed access policy | Unlisted inter-zone traffic and unsolicited Internet ingress denied by default. No Internet-facing RDP, SMB, LDAP, SQL or appliance management.
---

In [12]:
# test rag retrieval from internal environment data and external SMB security guidance

# from the Northbridge environment
test1 = rag_search(
    query="FILE04 SMB signing",
    collection="environment"
)
print(test1)

# from the external threat intelligence on SMB relay attack prerequisites
test2 = rag_search(
    query="SMB relay attack prerequisites",
    collection="guidance"
)
print(test2)

{'status': 'success', 'query': 'FILE04 SMB signing', 'collection': 'environment', 'results': [{'text': 'W05 File servers | 4 | Department shares and shared documents. SMB 3; SMB1 disabled. Finance and HR shares use AD groups. SMB signing required on three hosts; not required on FILE04.', 'source': 'Windows_Enterprise_Environment.docx', 'source_mode': 'simulated', 'relevance_score': 0.4045}, {'text': 'S03 control and exception register | SMB signing is not required on FILE04 within W05; SMB1 disabled across W05. Twelve E02 sensors stale; ten E02 devices have local admin exception. Overlap between these subsets is unknown.', 'source': 'Windows_Enterprise_Environment.docx', 'source_mode': 'simulated', 'relevance_score': 0.2964}, {'text': 'Could a reported SMB-related attack apply to FILE04, given signing is not required and SMB1 is disabled? What would access to its data imply? Could a VPN advisory affect N02 when firmware is unknown? What additional evidence is decisive? How do W15 servi

In [13]:
# add the tool to the llm's toolbox

llm_with_tools = llm.bind_tools([rag_search])

response = llm_with_tools.invoke(
    "Investigate the SMB signing weakness on FILE04. "
    "Retrieve both Northbridge environment evidence and "
    "external Microsoft SMB security guidance using rag_search."
)

print(response.tool_calls)

[{'name': 'rag_search', 'args': {'collection': 'environment', 'query': 'FILE04 SMB signing weakness evidence', 'top_k': 5}, 'id': 'a305604b-f94b-4315-8b87-e4ded5bc5355', 'type': 'tool_call'}]


In [14]:
# this function was coded with assistance from ChatGPT 6.0 on 10/12/2026 at 5:00PM.
# We defined the function's purpose, inputs, expected output. We added the try-except block to improve error handling.

def lookup_cisa_kev(cve_id: str) -> dict:
    """
    Check whether a CVE is listed in CISA's
    Known Exploited Vulnerabilities (KEV) catalog.

    Args:
        cve_id: CVE identifier to look up (e.g., CVE-2021-44228).

    Returns:
        Dictionary containing the lookup status, matching CVE details
        if found, and source attribution from CISA KEV.
    """

    url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

    cve_id = cve_id.strip().upper()

    # Basic input validation
    if not cve_id.startswith("CVE-"):
        return {
            "status": "error",
            "message": "Invalid CVE ID."
        }

    try:
        # Retrieve CISA KEV catalog
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        catalog = response.json()

        # Find matching CVE
        for vulnerability in catalog["vulnerabilities"]:
            if vulnerability["cveID"] == cve_id:
                return {
                    "status": "found",
                    "cve_id": cve_id,
                    "details": vulnerability,
                    "source": url,
                    "source_mode": "live",
                    "retrieved_at": datetime.now(timezone.utc).isoformat()
                }

        return {
            "status": "not_listed",
            "cve_id": cve_id,
            "message": "CVE not listed in CISA KEV.",
            "source": url,
            "source_mode": "live",
            "retrieved_at": datetime.now(timezone.utc).isoformat()
        }

    except (requests.RequestException, ValueError, KeyError) as e:
        return {
            "status": "unavailable",
            "cve_id": cve_id,
            "message": str(e),
            "source": url
        }


In [15]:
# test CISA KEV lookup tool with a known exploited CVE
result = lookup_cisa_kev("CVE-2021-44228")

print(json.dumps(result, indent=2))

{
  "status": "found",
  "cve_id": "CVE-2021-44228",
  "details": {
    "cveID": "CVE-2021-44228",
    "vendorProject": "Apache",
    "product": "Log4j2",
    "vulnerabilityName": "Apache Log4j2 Remote Code Execution Vulnerability",
    "dateAdded": "2021-12-10",
    "shortDescription": "Apache Log4j2 contains a vulnerability where JNDI features do not protect against attacker-controlled JNDI-related endpoints, allowing for remote code execution.",
    "requiredAction": "For all affected software assets for which updates exist, the only acceptable remediation actions are: 1) Apply updates; OR 2) remove affected assets from agency networks. Temporary mitigations using one of the measures provided at https://www.cisa.gov/uscert/ed-22-02-apache-log4j-recommended-mitigation-measures are only acceptable until updates are available.",
    "dueDate": "2021-12-24",
    "knownRansomwareCampaignUse": "Known",
    "forensicTriage": "No",
    "notes": "https://nvd.nist.gov/vuln/detail/CVE-2021-442

In [16]:
# this function was coded with assistance from ChatGPT 6.0 on 10/12/2026 at 5:00PM. We defined the function's purpose, inputs, and expected output.
# We rewrote refined the logic to distinguish confirmed, absent, and unknown attack prerequisites using True, False, and None
# Therefore, missing evidence is flagged rather than assumed.

# Convert string inputs from the LLM into Python values

def check_attack_applicability(
    smb_signing_not_required: bool,
    smb_reachable: bool | None = None,
    relay_prerequisites_met: bool | None = None,
    unsigned_connection_possible: bool | None = None
) -> dict:
    """
    Evaluate whether an SMB relay attack is applicable based on
    the available environment evidence and attack prerequisites.

    Args:
        smb_signing_not_required: Whether SMB signing is not required.
        smb_reachable: Whether an attacker can reach the SMB service.
        relay_prerequisites_met: Whether authentication relay prerequisites
            are satisfied.
        unsigned_connection_possible: Whether an unsigned SMB connection
            can be negotiated.

    Returns:
        Dictionary containing the applicability status, evaluated
        conditions, missing evidence, and analyst review requirement.

    Notes:
        True indicates a confirmed condition, False indicates a
        confirmed absent condition, and None indicates unknown.
        Missing evidence results in an insufficient_evidence status
        rather than an assumed conclusion.
    """

    # Convert string inputs from the LLM into Python values
    def normalize(value):
        if isinstance(value, str):
            if value.lower() == "unknown":
                return None
            if value.lower() == "true":
                return True
            if value.lower() == "false":
                return False
            raise ValueError(f"Invalid condition value: {value}")

        if value is None or type(value) is bool:
            return value

        raise ValueError(f"Invalid condition value: {value}")

    smb_signing_not_required = normalize(smb_signing_not_required)
    smb_reachable = normalize(smb_reachable)
    relay_prerequisites_met = normalize(relay_prerequisites_met)
    unsigned_connection_possible = normalize(unsigned_connection_possible)

    conditions = {
        "SMB signing not required": smb_signing_not_required,
        "SMB reachable by attacker": smb_reachable,
        "Relay prerequisites met": relay_prerequisites_met,
        "Unsigned connection possible": unsigned_connection_possible
    }

    # Identify unknown conditions
    unknowns = [
        condition
        for condition, value in conditions.items()
        if value is None
    ]

    # Determine applicability
    if any(value is False for value in conditions.values()):
        status = "not_applicable"

    elif unknowns:
        status = "insufficient_evidence"

    else:
        status = "applicable"

    return {
        "attack": "SMB relay attack",
        "asset_id": "FILE04",
        "applicability": status,
        "conditions": conditions,
        "unknowns": unknowns,
        "requires_review": status == "insufficient_evidence"
    }

In [17]:
# test SMB relay applicability tool using known Northbridge evidence
result = check_attack_applicability(
    smb_signing_not_required=True,
    smb_reachable=None,
    relay_prerequisites_met=None,
    unsigned_connection_possible=None
)

print(json.dumps(result, indent=2))

{
  "attack": "SMB relay attack",
  "asset_id": "FILE04",
  "applicability": "insufficient_evidence",
  "conditions": {
    "SMB signing not required": true,
    "SMB reachable by attacker": null,
    "Relay prerequisites met": null,
    "Unsigned connection possible": null
  },
  "unknowns": [
    "SMB reachable by attacker",
    "Relay prerequisites met",
    "Unsigned connection possible"
  ],
  "requires_review": true
}


In [18]:
# bind all tools to Ollama
llm_with_tools = llm.bind_tools([
    rag_search,
    lookup_cisa_kev,
    check_attack_applicability
])

In [19]:
# this function was coded with assistance from ChatGPT 6.0 on 10/12/2026 at 5:00PM. We defined the function's purpose, inputs, and expected output.
# We added a risk-rating policy to assign High, Medium, or Low based on attack applicability and business impact.

# this tool generates a structured cybersecurity risk assessment, given the asset_id, threat, applicability, business impact, rationale, evidence, unknowns, and reccomended action.

def generate_report(
    asset_id: str,
    threat: str,
    applicability: str,
    business_impact: str,
    rationale: str,
    evidence: list,
    unknowns: list,
    recommended_action: str,
    impact_provisional: bool = True,
    evidence_needed: list = None
) -> dict:
    """
    Generate a structured cybersecurity risk assessment.

    Args:
        asset_id: Asset being assessed (e.g., FILE04).
        threat: Description of the threat or weakness.
        applicability: applicable, not_applicable, or insufficient_evidence.
        business_impact: critical, departmental, or minor.
        rationale: Explanation of the risk decision.
        evidence: Supporting evidence with source references.
        unknowns: Missing or uncertain facts.
        recommended_action: Recommended remediation.
        impact_provisional: Whether impact requires further verification.
        evidence_needed: Additional evidence that could change the decision.

    Returns:
        Structured risk assessment.
    """

    # Normalize common input variations from the LLM
    applicability = applicability.strip().lower().replace(" ", "_")

    if applicability == "notapplicable":
        applicability = "not_applicable"

    business_impact = business_impact.strip().lower()

    impact_aliases = {
        "moderate": "departmental",
        "medium": "departmental",
        "high": "critical",
        "low": "minor"
    }

    business_impact = impact_aliases.get(
        business_impact, business_impact
    )

    # Validate inputs
    valid_applicability = [
        "applicable",
        "not_applicable",
        "insufficient_evidence"
    ]

    valid_impacts = ["critical", "departmental", "minor"]

    if applicability not in valid_applicability:
        return {"status": "error", "message": "Invalid applicability"}

    if business_impact not in valid_impacts:
        return {"status": "error", "message": "Invalid business impact"}

    # Require supporting evidence before assigning a risk rating
    if not evidence:
        risk_rating = "Insufficient Evidence"

    elif applicability == "insufficient_evidence":
        risk_rating = "Insufficient Evidence"

    elif applicability == "not_applicable":
        risk_rating = "Not Applicable"

    elif business_impact == "critical":
        risk_rating = "High"

    elif business_impact == "departmental":
        risk_rating = "Medium"

    else:
        risk_rating = "Low"

    # Identify evidence needed to resolve uncertainty
    if evidence_needed is None:
        evidence_needed = unknowns.copy()

    # Generate structured assessment
    report = {
        "asset_id": asset_id,
        "threat": threat,

        "applicability": {
            "status": applicability,
            "evidence_sufficient": applicability != "insufficient_evidence"
        },

        "business_impact": {
            "level": business_impact,
            "provisional": impact_provisional
        },

        "risk_rating": risk_rating,
        "rating_policy": "Northbridge applicability and impact policy",
        "rationale": rationale,

        "evidence": evidence,
        "unknowns": unknowns,
        "evidence_needed": evidence_needed,

        "recommended_action": recommended_action,
        "analyst_review": "pending",
        "generated_at": datetime.now(timezone.utc).isoformat()
    }

    # Save assessment as JSON
    with open("risk_assessment.json", "w") as file:
        json.dump(report, file, indent=4)

    return report



In [20]:
# test the Data Synthesizer / Reporter tool
result = generate_report(
    asset_id="FILE04",
    threat="SMB signing weakness and potential SMB relay attack",
    applicability="insufficient_evidence",
    business_impact="departmental",
    rationale=(
        "FILE04 does not require SMB signing (W05, S03). "
        "However, internal SMB reachability and authentication "
        "relay prerequisites are not confirmed. The W05 asset group "
        "supports departmental file sharing, but the exact contents "
        "of FILE04 are unknown."
    ),
    evidence=[
        {
            "source_id": "S03",
            "source_mode": "simulated",
            "fact": "SMB signing is not required on FILE04."
        },
        {
            "source_id": "S02",
            "source_mode": "simulated",
            "fact": "No Internet-facing SMB and default-deny inter-zone policy."
        },
        {
            "source_id": "W05",
            "source_mode": "simulated",
            "fact": "File-server group supports departmental shares."
        }
    ],
    unknowns=[
        "Internal SMB reachability to FILE04",
        "Authentication relay prerequisites",
        "Effective SMB signing negotiation",
        "Exact data sensitivity of FILE04"
    ],
    recommended_action=(
        "Verify FILE04's effective SMB signing configuration "
        "and network access paths. Require SMB signing where "
        "appropriate and validate departmental share permissions."
    )
)

print(json.dumps(result, indent=4))

{
    "asset_id": "FILE04",
    "threat": "SMB signing weakness and potential SMB relay attack",
    "applicability": {
        "status": "insufficient_evidence",
        "evidence_sufficient": false
    },
    "business_impact": {
        "level": "departmental",
        "provisional": true
    },
    "risk_rating": "Insufficient Evidence",
    "rating_policy": "Northbridge applicability and impact policy",
    "rationale": "FILE04 does not require SMB signing (W05, S03). However, internal SMB reachability and authentication relay prerequisites are not confirmed. The W05 asset group supports departmental file sharing, but the exact contents of FILE04 are unknown.",
    "evidence": [
        {
            "source_id": "S03",
            "source_mode": "simulated",
            "fact": "SMB signing is not required on FILE04."
        },
        {
            "source_id": "S02",
            "source_mode": "simulated",
            "fact": "No Internet-facing SMB and default-deny inter-zone

In [21]:
# bind all tools to Ollama
llm_with_tools = llm.bind_tools([
    rag_search,
    lookup_cisa_kev,
    check_attack_applicability,
    generate_report
])

In [22]:
# map tool names to Python functions
tools = {
    "rag_search": rag_search,
    "lookup_cisa_kev": lookup_cisa_kev,
    "check_attack_applicability": check_attack_applicability,
    "generate_report": generate_report
}

In [23]:
system_prompt = """
You are a Cyber Threat Intelligence risk assessment agent
for Northbridge Services.

Your task is to assess the SMB signing weakness on FILE04.

Follow this workflow:

1. Use rag_search with collection="environment" to retrieve
   FILE04 configuration, network controls, and business impact.

2. Use rag_search with collection="guidance" to retrieve
   Microsoft SMB signing and relay attack guidance.

3. Only use lookup_cisa_kev if a specific CVE is relevant.

4. Use check_attack_applicability to evaluate whether the
   available evidence supports the SMB relay attack prerequisites.

5. Use generate_report to produce the final JSON assessment.

Important rules:
- Never invent evidence or source references.
- Northbridge environment information is simulated.
- Distinguish a confirmed weakness from a confirmed attack path.
- Do not assume FILE04 contains Finance or HR data.
- Do not assume SMB access requires no credentials.
- Missing decisive evidence must produce Insufficient Evidence.
- Apply the group's rating policy consistently.
- Clearly identify unknowns and recommend next steps.
- Do not invent CVE identifiers.

EVIDENCE REQUIREMENTS:
- FILE04 does not require SMB signing (confirmed by W05/S03).
- Internet-facing SMB is absent (S02).
- Internal attacker reachability to FILE04 is unknown.
- Authentication relay prerequisites are unknown.
- Whether an unsigned connection can actually be
  negotiated is unknown.
- The exact sensitivity of files on FILE04 is unknown.

Do not convert unknown prerequisites into False or True.
Use null/None or "unknown" where evidence is missing.

REPORT TOOL REQUIREMENTS:
- applicability must be exactly:
  applicable, not_applicable, or insufficient_evidence
- business_impact must be exactly:
  critical, departmental, or minor
- Put explanations in rationale, not categorical fields.
- Preserve the applicability checker's result.

REPORT ACCURACY REQUIREMENTS:

1. Use precise terminology: SMB signing is "not required"
   on FILE04, not necessarily "disabled" or "unenforced".
   This does not prove that all SMB sessions are unsigned.

2. Do not assume FILE04 specifically contains Finance or HR
   data. Treat departmental business impact as provisional
   until data classification and share permissions are verified.

3. Preserve original evidence IDs (W05, S02, S03),
   document references, and external source URLs.
   Do not attribute Northbridge configuration facts to Microsoft.

4. Do not claim that requiring SMB signing eliminates all
   possible relay or authentication attacks.

5. Include FILE04's data classification, share permissions,
   internal SMB reachability, and relay prerequisites among
   the unknowns when not established by evidence.

6. Clearly distinguish confirmed facts, assumptions,
   and missing evidence. Never invent supporting evidence.

7. Recommend reviewing firewall rules, SMB configuration,
   and authorized connectivity testing rather than assuming
   a live network scan is necessary.

WORKFLOW COMPLETION REQUIREMENT:

You MUST call generate_report after completing the
environment lookup, guidance lookup, and applicability check.

Do not finish with a text-only response.

The assessment is incomplete until generate_report
successfully returns a risk_rating.

When calling generate_report:
- applicability must be exactly:
  applicable, not_applicable, insufficient_evidence
- business_impact must be exactly:
  critical, departmental, minor

Put explanations in rationale, not categorical fields.
"""

In [24]:
# this function will run the agentic risk assessment workflow by
# processing the user's query, invoking the appropriate tools,
# collecting evidence, and generating a final risk assessment report.

def run_agent(user_query: str):
    """
    Run the security triage agent using Ollama and Python tools.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_query)
    ]

    # Track evidence retrieval and tool usage
    retrieved_collections = set()
    tool_history = []

    for step in range(10):

        # Ask Ollama what to do next
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If Ollama is finished
        if not response.tool_calls:
            print("\nAGENT RESPONSE:")
            print(response.content)
            break

        # Execute requested tools
        for call in response.tool_calls:

            tool_name = call["name"]
            tool_args = call["args"]

            print(f"\nTool: {tool_name}")
            print(f"Arguments: {tool_args}")

            try:
                if tool_name not in tools:
                    raise ValueError("Unknown tool requested")

                # Do not generate a report before evidence lookups
                if tool_name == "generate_report" and not {
                    "environment", "guidance"
                }.issubset(retrieved_collections):
                    result = {
                        "status": "error",
                        "message": "Retrieve environment and guidance evidence first."
                    }

                else:
                    result = tools[tool_name](**tool_args)

                if (
                    tool_name == "rag_search"
                    and result.get("status") == "success"
                    and result.get("results")
                ):
                    retrieved_collections.add(
                        tool_args.get("collection", "environment")
                    )

            except Exception as e:
                result = {
                    "status": "error",
                    "message": str(e)
                }

            # Save request and response for demonstration
            tool_history.append({
                "tool": tool_name,
                "arguments": tool_args,
                "result": result
            })

            # Return tool output to Ollama
            messages.append(
                ToolMessage(
                    content=json.dumps(result, default=str),
                    tool_call_id=call["id"]
                )
            )

            # Stop the agent after successfully generating the final report
            if tool_name == "generate_report" and "risk_rating" in result:
              print("\nSUCCESS: Final risk assessment generated!")
              print(json.dumps(result, indent=2))
              return messages, tool_history

    else:
        print("Agent stopped: maximum iterations reached.")

    return messages, tool_history


In [25]:
# run a complete SMB Signing risk assessment

messages, tool_history = run_agent(
    "Assess the risk of SMB signing not being required "
    "on FILE04 in Northbridge Services. "
    "Investigate applicability, network controls, "
    "business impact, and recommended actions."
)


Tool: rag_search
Arguments: {'collection': 'environment', 'query': 'FILE04 SMB signing configuration network controls business impact', 'top_k': 5}

Tool: rag_search
Arguments: {'collection': 'guidance', 'query': 'Microsoft SMB signing relay attack guidance', 'top_k': 5}

Tool: check_attack_applicability
Arguments: {'relay_prerequisites_met': None, 'smb_reachable': None, 'smb_signing_not_required': True, 'unsigned_connection_possible': None}

Tool: generate_report
Arguments: {'applicability': 'insufficient_evidence', 'asset_id': 'FILE04', 'business_impact': 'departmental', 'evidence': [{'description': 'File servers in W05: SMB signing required on three hosts; not required on FILE04.', 'id': 'W05', 'relevance': 'confirmed', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'No internet-facing SMB service on FILE04.', 'id': 'S02', 'relevance': 'confirmed', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'SMB signing is not required on FILE04 within W05;

In [26]:
# display the generated risk assessment

with open("risk_assessment.json", "r") as file:
    assessment = json.load(file)

print(json.dumps(assessment, indent=2))

{
  "asset_id": "FILE04",
  "threat": "SMB signing not required on FILE04 potentially allows SMB relay attacks if an attacker can reach the host, negotiate an unsigned connection, and supply valid credentials or session keys. Without signing, the integrity of SMB traffic cannot be verified, making it easier for an attacker to intercept or modify SMB messages.",
  "applicability": {
    "status": "insufficient_evidence",
    "evidence_sufficient": false
  },
  "business_impact": {
    "level": "departmental",
    "provisional": true
  },
  "risk_rating": "Insufficient Evidence",
  "rating_policy": "Northbridge applicability and impact policy",
  "rationale": "SMB signing is not required on FILE04, which removes a layer of integrity protection that helps prevent SMB relay attacks. However, the absence of SMB signing alone does not confirm that a relay attack is possible. Key prerequisites\u2014such as internal attacker reachability, the ability to negotiate an unsigned connection, and th

In [27]:
# save tool execution history

with open("tool_history.json", "w") as file:
    json.dump(tool_history, file, indent=2, default=str)

print("Tool history saved.")

Tool history saved.


# Tool Evaluation

In [28]:
# Test 1: Verify attack applicability logic

applicability_tests = [
    {
        "name": "All prerequisites confirmed",
        "inputs": (True, True, True, True),
        "expected": "applicable"
    },
    {
        "name": "Network access blocked",
        "inputs": (True, False, True, True),
        "expected": "not_applicable"
    },
    {
        "name": "Network access unknown",
        "inputs": (True, None, True, True),
        "expected": "insufficient_evidence"
    }
]

for test in applicability_tests:
    result = check_attack_applicability(*test["inputs"])
    actual = result["applicability"]

    print(test["name"])
    print("Expected:", test["expected"])
    print("Actual:", actual)
    print("PASS" if actual == test["expected"] else "FAIL")
    print("-" * 40)

All prerequisites confirmed
Expected: applicable
Actual: applicable
PASS
----------------------------------------
Network access blocked
Expected: not_applicable
Actual: not_applicable
PASS
----------------------------------------
Network access unknown
Expected: insufficient_evidence
Actual: insufficient_evidence
PASS
----------------------------------------


In [29]:
# Test 2: Verify risk ratings across business impact levels

from pathlib import Path

report_path = Path("risk_assessment.json")
original_report = report_path.read_bytes() if report_path.exists() else None

impact_tests = [
    ("critical", "High"),
    ("departmental", "Medium"),
    ("minor", "Low")
]

try:
    for impact, expected in impact_tests:

        result = generate_report(
            asset_id="TEST-FILE04",
            threat="Simulated SMB relay attack",
            applicability="applicable",
            business_impact=impact,
            rationale="Controlled test with confirmed applicability.",
            evidence=[
                {
                    "source_id": "TEST-001",
                    "source_mode": "simulated",
                    "fact": "All prerequisites confirmed in test fixture."
                }
            ],
            unknowns=[],
            recommended_action="Review and remediate SMB configuration."
        )

        actual = result["risk_rating"]

        print(f"Impact: {impact}")
        print(f"Expected: {expected}")
        print(f"Actual: {actual}")
        print("PASS" if actual == expected else "FAIL")
        print("-" * 40)

finally:
    if original_report is not None:
        report_path.write_bytes(original_report)
    else:
        report_path.unlink(missing_ok=True)

Impact: critical
Expected: High
Actual: High
PASS
----------------------------------------
Impact: departmental
Expected: Medium
Actual: Medium
PASS
----------------------------------------
Impact: minor
Expected: Low
Actual: Low
PASS
----------------------------------------


In [30]:
# Test 3: Verify handling of insufficient evidence

result = check_attack_applicability(
    smb_signing_not_required=True,
    smb_reachable=None,
    relay_prerequisites_met=None,
    unsigned_connection_possible=None
)

expected = "insufficient_evidence"
actual = result["applicability"]

print("Expected:", expected)
print("Actual:", actual)
print("PASS" if actual == expected else "FAIL")
print("Missing evidence:", result["unknowns"])

Expected: insufficient_evidence
Actual: insufficient_evidence
PASS
Missing evidence: ['SMB reachable by attacker', 'Relay prerequisites met', 'Unsigned connection possible']


In [31]:
# Test 4: Simulate an unavailable CISA KEV API

from unittest.mock import patch
import requests

with patch(
    "requests.get",
    side_effect=requests.ConnectionError("Simulated API failure")
):
    result = lookup_cisa_kev("CVE-2021-44228")

expected = "unavailable"
actual = result["status"]

print("Expected:", expected)
print("Actual:", actual)
print("PASS" if actual == expected else "FAIL")
print("Error:", result.get("message"))

Expected: unavailable
Actual: unavailable
PASS
Error: Simulated API failure


# Agent Evaluation

In [32]:
# assess FILE04 using the complete agent

messages, tool_history = run_agent(
    "Assess the SMB signing weakness on FILE04 at Northbridge Services. "
    "Retrieve internal environment evidence and external Microsoft guidance, "
    "evaluate attack applicability and business impact, "
    "and generate an evidence-supported risk assessment."
)


Tool: rag_search
Arguments: {'collection': 'environment', 'query': 'FILE04 SMB signing configuration, network controls, business impact', 'top_k': 5}

Tool: rag_search
Arguments: {'collection': 'guidance', 'query': 'Microsoft SMB signing guidance relay attack', 'top_k': 5}

Tool: check_attack_applicability
Arguments: {'relay_prerequisites_met': None, 'smb_reachable': None, 'smb_signing_not_required': True, 'unsigned_connection_possible': None}

Tool: generate_report
Arguments: {'applicability': 'insufficient_evidence', 'asset_id': 'FILE04', 'business_impact': 'departmental', 'evidence': [{'description': 'File servers in W05: SMB signing not required on FILE04', 'id': 'W05', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'No internet-facing SMB service on FILE04', 'id': 'S02', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'SMB signing not required on FILE04 within W05', 'id': 'S03', 'source': 'Windows_Enterprise_Environment.docx'}], 'evidence_need

In [39]:
# inspect tool execution history

for i, entry in enumerate(tool_history, 1):
    tool_name = entry["tool"]
    result = entry["result"]

    if result.get("status") == "error":
        status = "ERROR"
    elif tool_name == "rag_search":
        status = result.get("status", "Unknown")
    elif tool_name == "check_attack_applicability":
        status = result.get("applicability", "Unknown")
    elif tool_name == "generate_report":
        status = result.get("risk_rating", "Unknown")
    elif tool_name == "lookup_cisa_kev":
        status = result.get("status", "Unknown")
    else:
        status = "Unknown"

    print(f"{i}. Tool: {tool_name}")
    print(f"   Status: {status}")
    print("-" * 40)

1. Tool: rag_search
   Status: success
----------------------------------------
2. Tool: rag_search
   Status: success
----------------------------------------
3. Tool: check_attack_applicability
   Status: insufficient_evidence
----------------------------------------
4. Tool: generate_report
   Status: Insufficient Evidence
----------------------------------------


In [34]:
# Inspect errors returned by the report tool

for entry in tool_history:
    if entry["tool"] == "generate_report":
        print("Arguments:", entry["arguments"])
        print("Result:", entry["result"])
        print("-" * 50)

Arguments: {'applicability': 'insufficient_evidence', 'asset_id': 'FILE04', 'business_impact': 'departmental', 'evidence': [{'description': 'File servers in W05: SMB signing not required on FILE04', 'id': 'W05', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'No internet-facing SMB service on FILE04', 'id': 'S02', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'SMB signing not required on FILE04 within W05', 'id': 'S03', 'source': 'Windows_Enterprise_Environment.docx'}], 'evidence_needed': [{'description': 'Internal attacker reachability to FILE04'}, {'description': 'Authentication relay prerequisites (e.g., presence of SMB relay services, valid credentials, etc.)'}, {'description': 'Whether an unsigned SMB connection can actually be negotiated'}, {'description': 'Data classification and share permissions of FILE04'}], 'impact_provisional': True, 'rationale': 'The evidence confirms that SMB signing is not required on FILE04 (W05, S03) and that ther

# Save Agent State and Files

In [35]:
# Save successful agent execution history
with open("tool_history.json", "w") as file:
    json.dump(tool_history, file, indent=2, default=str)

print("Agent execution history saved.")

Agent execution history saved.


In [36]:
# save the completed agent run for reproducibility

baseline = {
    "case": "FILE04 SMB Signing Weakness",
    "model": "gpt-oss:20b",
    "provider": "Ollama Cloud",
    "temperature": 0,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "environment": "Northbridge Services - Simulated",
    "environment_file": filename,
    "system_prompt": system_prompt,
    "tool_history": tool_history
}

with open("agent_baseline.json", "w") as file:
    json.dump(baseline, file, indent=4, default=str)

print("Agent baseline saved!")

Agent baseline saved!


In [37]:
from google.colab import files

# Download the final risk assessment
files.download("risk_assessment.json")

# Download the agent execution history
files.download("agent_baseline.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
# Save retrieved Microsoft SMB guidance
with open("microsoft_smb_guidance.json", "w") as file:
    json.dump(guidance_passages, file, indent=4)

files.download("microsoft_smb_guidance.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Validating if the Agent uses the CISA KEV tool

In [40]:
result = lookup_cisa_kev("CVE-2025-33073")
print(json.dumps(result, indent=2))

{
  "status": "found",
  "cve_id": "CVE-2025-33073",
  "details": {
    "cveID": "CVE-2025-33073",
    "vendorProject": "Microsoft",
    "product": "Windows",
    "vulnerabilityName": "Microsoft Windows SMB Client Improper Access Control Vulnerability",
    "dateAdded": "2025-10-20",
    "shortDescription": "Microsoft Windows SMB Client contains an improper access control vulnerability that could allow for privilege escalation. An attacker could execute a specially crafted malicious script to coerce the victim machine to connect back to the attack system using SMB and authenticate.",
    "requiredAction": "Apply mitigations per vendor instructions, follow applicable BOD 22-01 guidance for cloud services, or discontinue use of the product if mitigations are unavailable.",
    "dueDate": "2025-11-10",
    "knownRansomwareCampaignUse": "Unknown",
    "forensicTriage": "No",
    "notes": "https://msrc.microsoft.com/update-guide/en-US/advisory/CVE-2025-33073 ; https://nvd.nist.gov/vuln/deta

In [41]:
messages, tool_history = run_agent(
    "Assess the SMB signing weakness on FILE04 at Northbridge Services. "
    "Also investigate CVE-2025-33073 using lookup_cisa_kev. "
    "Determine whether the CVE is applicable based on the available "
    "environment evidence. Do not assume FILE04 is vulnerable simply "
    "because SMB signing is not required."
)


Tool: rag_search
Arguments: {'collection': 'environment', 'query': 'FILE04 configuration SMB signing not required W05 S03 network controls S02 business impact', 'top_k': 5}

Tool: rag_search
Arguments: {'collection': 'guidance', 'query': 'Microsoft SMB signing relay attack guidance', 'top_k': 5}

Tool: lookup_cisa_kev
Arguments: {'cve_id': 'CVE-2025-33073'}

Tool: check_attack_applicability
Arguments: {'relay_prerequisites_met': 'unknown', 'smb_reachable': 'unknown', 'smb_signing_not_required': True, 'unsigned_connection_possible': 'unknown'}

Tool: generate_report
Arguments: {'applicability': 'insufficient_evidence', 'asset_id': 'FILE04', 'business_impact': 'departmental', 'evidence': [{'description': 'SMB signing not required on FILE04 within W05; SMB1 disabled across W05.', 'id': 'W05', 'source': 'Windows_Enterprise_Environment.docx'}, {'description': 'Control and exception register indicates SMB signing is not required on FILE04.', 'id': 'S03', 'source': 'Windows_Enterprise_Enviro

In [42]:
for entry in tool_history:
    print(entry["tool"])

rag_search
rag_search
lookup_cisa_kev
check_attack_applicability
generate_report
